### Combine all cvs files in one file 

In [1]:
# Cell 1 - Imports
import pandas as pd
import glob
import os

print("✅ Libraries imported!")

✅ Libraries imported!


In [2]:
# Cell 2 - Define Paths
base_path      = "D:/Python/Lithofacies_Classification_NMM"
raw_path       = os.path.join(base_path, "data/raw")
processed_path = os.path.join(base_path, "data/processed")

print(f"📂 Raw data folder      : {raw_path}")
print(f"📂 Processed data folder: {processed_path}")

📂 Raw data folder      : D:/Python/Lithofacies_Classification_NMM\data/raw
📂 Processed data folder: D:/Python/Lithofacies_Classification_NMM\data/processed


In [3]:
# Cell 3 - Load All CSV Files & Inspect
csv_files = glob.glob(os.path.join(raw_path, "*.csv"))
print(f"Found {len(csv_files)} CSV files:\n")

for file in csv_files:
    df = pd.read_csv(file)
    has_label = "FORCE_2020_LITHOFACIES_LITHOLOGY" in df.columns
    label_tag = "✅ LABELED  " if has_label else "⚠️  UNLABELED"
    print(f"  {label_tag} → {os.path.basename(file):35s} | Shape: {df.shape}")

Found 21 CSV files:

  ✅ LABELED   → 15_9-23.csv                         | Shape: (11063, 29)
  ⚠️  UNLABELED → 15_9_14_well.csv                    | Shape: (20281, 27)
  ✅ LABELED   → 16_2-7.csv                          | Shape: (11683, 29)
  ✅ LABELED   → 16_7-6.csv                          | Shape: (10222, 29)
  ✅ LABELED   → 17_4-1.csv                          | Shape: (17271, 29)
  ✅ LABELED   → 25_10-9.csv                         | Shape: (10788, 29)
  ⚠️  UNLABELED → 25_10_10_well.csv                   | Shape: (9320, 27)
  ⚠️  UNLABELED → 25_11_24_well.csv                   | Shape: (4195, 27)
  ⚠️  UNLABELED → 25_5_3_well.csv                     | Shape: (11324, 27)
  ⚠️  UNLABELED → 29_3_1_well.csv                     | Shape: (24213, 27)
  ✅ LABELED   → 31_2-10.csv                         | Shape: (9033, 29)
  ✅ LABELED   → 31_2-21 S.csv                       | Shape: (7840, 29)
  ✅ LABELED   → 31_6-5.csv                          | Shape: (11597, 20)
  ✅ LABELED   → 31_6-8.c

In [4]:
# Cell 4 - Combine Files
labeled_list   = []
unlabeled_list = []

for file in csv_files:
    df       = pd.read_csv(file)
    filename = os.path.basename(file)
    
    # Tag each row with its source well file
    df["SOURCE_FILE"] = filename
    
    if "FORCE_2020_LITHOFACIES_LITHOLOGY" in df.columns:
        labeled_list.append(df)
    else:
        unlabeled_list.append(df)

# Merge
labeled_df   = pd.concat(labeled_list,   ignore_index=True) if labeled_list   else None
unlabeled_df = pd.concat(unlabeled_list, ignore_index=True) if unlabeled_list else None

print(f"{'='*55}")
print(f"  Total files        : {len(csv_files)}")
print(f"  Labeled files      : {len(labeled_list)}")
print(f"  Unlabeled files    : {len(unlabeled_list)}")
print(f"  Labeled   shape    : {labeled_df.shape}")
if unlabeled_df is not None:
    print(f"  Unlabeled shape    : {unlabeled_df.shape}")
print(f"{'='*55}")

  Total files        : 21
  Labeled files      : 11
  Unlabeled files    : 10
  Labeled   shape    : (122117, 32)
  Unlabeled shape    : (136786, 28)


In [5]:
# Cell 5 - Target Class Distribution
print("🎯 Lithology Class Distribution:\n")
class_counts = labeled_df["FORCE_2020_LITHOFACIES_LITHOLOGY"].value_counts()
print(class_counts)
print(f"\nTotal unique classes: {class_counts.shape[0]}")

🎯 Lithology Class Distribution:

FORCE_2020_LITHOFACIES_LITHOLOGY
65000.0    69305
30000.0    14794
65030.0    10507
70000.0     8721
88000.0     6498
80000.0     5266
70032.0     2905
99000.0     2366
86000.0      597
74000.0      269
90000.0      196
Name: count, dtype: int64

Total unique classes: 11


In [6]:
# Cell 6 - Save to Processed Folder
labeled_path = os.path.join(processed_path, "labeled_data.csv")
labeled_df.to_csv(labeled_path, index=False)
print(f"💾 Saved → {labeled_path}")

if unlabeled_df is not None:
    unlabeled_path = os.path.join(processed_path, "unlabeled_data.csv")
    unlabeled_df.to_csv(unlabeled_path, index=False)
    print(f"💾 Saved → {unlabeled_path}")

print("\n✅ Data combination complete!")

💾 Saved → D:/Python/Lithofacies_Classification_NMM\data/processed\labeled_data.csv
💾 Saved → D:/Python/Lithofacies_Classification_NMM\data/processed\unlabeled_data.csv

✅ Data combination complete!


#### Class imblance - This imbalance will hurt accuracy on rare classes — we'll fix it with SMOTE + class weights later.
##### 57% is 65000 (shale)


# Generate Test file here 

In [1]:
# Generate Random Test Data
import pandas as pd
import numpy as np
import os

TEST_PATH = "D:/Python/Lithofacies_Classification_NMM/data/test"
os.makedirs(TEST_PATH, exist_ok=True)

np.random.seed(42)

# Known wells, formations, groups from training data
WELLS      = ["15/9-23", "16/2-7", "16/7-6",
              "17/4-1",  "25/10-9"]
FORMATIONS = ["Draupne", "Skagerrak", "Statfjord",
              "Brent",   "Dunlin",    "Heather",
              "Sleipner","Tor",       "Ekofisk"]
GROUPS     = ["VIKING", "STATFJORD", "BRENT",
              "DUNLIN", "ZECHSTEIN", "ROTLIEGEND"]

# Generate 3 test files simulating different wells
files = {
    "test_well_A.csv" : 500,
    "test_well_B.csv" : 300,
    "test_well_C.csv" : 800,
}

for filename, n_rows in files.items():

    # Simulate realistic depth range
    depth_start = np.random.randint(800,  2000)
    depth_end   = depth_start + np.random.randint(500, 2500)
    depths      = np.linspace(depth_start, depth_end, n_rows)

    df_test = pd.DataFrame({
        # Depth
        "DEPTH_MD"  : depths.round(3),

        # Gamma Ray — log response (API)
        "GR"        : np.clip(
                        np.random.normal(75, 35, n_rows),
                        0, 300).round(3),

        # Compressional Sonic (us/ft)
        "DTC"       : np.clip(
                        np.random.normal(100, 25, n_rows),
                        40, 200).round(3),

        # Caliper (inches)
        "CALI"      : np.clip(
                        np.random.normal(12.5, 2, n_rows),
                        8, 22).round(3),

        # Deep Resistivity (ohm.m)
        "RDEP"      : np.clip(
                        np.random.lognormal(0.5, 1.0, n_rows),
                        0.1, 1000).round(3),

        # Neutron Porosity (v/v)
        "NPHI"      : np.clip(
                        np.random.normal(0.25, 0.12, n_rows),
                        0, 0.9).round(4),

        # Bulk Density (g/cc)
        "RHOB"      : np.clip(
                        np.random.normal(2.35, 0.25, n_rows),
                        1.0, 3.2).round(4),

        # Density Correction (g/cc)
        "DRHO"      : np.clip(
                        np.random.normal(0.01, 0.03, n_rows),
                        -0.1, 0.15).round(4),

        # Medium Resistivity (ohm.m)
        "RMED"      : np.clip(
                        np.random.lognormal(0.4, 0.9, n_rows),
                        0.1, 500).round(3),

        # Photoelectric Factor (b/e)
        "PEF"       : np.clip(
                        np.random.normal(2.5, 1.2, n_rows),
                        1.0, 6.0).round(3),

        # Categorical
        "WELL"      : np.random.choice(WELLS,      n_rows),
        "FORMATION" : np.random.choice(FORMATIONS, n_rows),
        "GROUP"     : np.random.choice(GROUPS,     n_rows),
    })

    # Add ~5% missing values to simulate real data
    for col in ["GR", "DTC", "NPHI", "RHOB", "RDEP"]:
        mask = np.random.random(n_rows) < 0.05
        df_test.loc[mask, col] = np.nan

    # Save
    out = f"{TEST_PATH}/{filename}"
    df_test.to_csv(out, index=False)

    print(f"✅ Generated : {filename}")
    print(f"   Rows      : {n_rows:,}")
    print(f"   Depth     : {depth_start:.0f}m — {depth_end:.0f}m")
    print(f"   Missing % : ~5% on GR/DTC/NPHI/RHOB/RDEP")
    print(f"   Saved     : data/test/{filename}\n")

print("=" * 50)
print(f"✅ All test files generated → data/test/")
print(f"   Total rows : {sum(files.values()):,}")
print(f"\n📋 Files created:")
for f in files:
    print(f"   → {f}")
print(f"\n🚀 Now run the batch prediction cell!")

✅ Generated : test_well_A.csv
   Rows      : 500
   Depth     : 1926m — 3885m
   Missing % : ~5% on GR/DTC/NPHI/RHOB/RDEP
   Saved     : data/test/test_well_A.csv

✅ Generated : test_well_B.csv
   Rows      : 300
   Depth     : 1158m — 1906m
   Missing % : ~5% on GR/DTC/NPHI/RHOB/RDEP
   Saved     : data/test/test_well_B.csv

✅ Generated : test_well_C.csv
   Rows      : 800
   Depth     : 1528m — 3922m
   Missing % : ~5% on GR/DTC/NPHI/RHOB/RDEP
   Saved     : data/test/test_well_C.csv

✅ All test files generated → data/test/
   Total rows : 1,600

📋 Files created:
   → test_well_A.csv
   → test_well_B.csv
   → test_well_C.csv

🚀 Now run the batch prediction cell!
